# 00 Data Cleaning

Before any exploration, statistics, or modelling, the raw extract is audited and cleaned in this notebook. Every later notebook (EDA, feature engineering, segmentation, forecasting, rating, basket) reads the artifact produced here and can assume zero NaN, parsed dates, and internally consistent monetary columns.

The order of work in this notebook:

1. Load the raw CSV and inspect schema and dtypes.
2. Audit missing values column by column.
3. Decide and document a per column cleaning policy.
4. Apply the policy and verify the result.
5. Quantify the impact of the cleaning choice on a downstream model, so the policy is justified by evidence rather than convenience.
6. Persist `data/processed/transactions_clean.parquet` for the rest of the pipeline.


In [1]:
import os, warnings
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config
from src.data_loader import load_raw, basic_clean


## 1. Load the raw CSV

In [2]:
raw = load_raw()
print('shape:', raw.shape)
raw.head()


shape: (1003, 17)


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7.0,26.1415,548.9715,1/5/19,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5.0,3.8200,80.2200,3/8/19,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7.0,16.2155,340.5255,3/3/19,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8.0,23.2880,489.0480,1/27/19,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7.0,30.2085,634.3785,2/8/19,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [3]:
raw.dtypes.to_frame('dtype')

,dtype
Invoice ID,object
Branch,object
City,object
Customer type,object
Gender,object
Product line,object
Unit price,float64
Quantity,float64
Tax 5%,float64
Total,float64


Two things to notice from the dtypes table:

- `Date` and `Time` are still `object` (strings). They must be parsed before any time series work.
- `Unit price` and `Quantity` are numeric, so any missing value will surface as NaN, not an empty string.


## 2. Missing value audit

In [4]:
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(raw) * 100).round(2)
audit = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
audit


,missing_count,missing_pct
Customer type,79,7.88
Product line,43,4.29
Quantity,20,1.99
Unit price,7,0.70


Five columns carry missing values, totalling around 15 percent of cells across the affected columns. A blanket `dropna()` on the whole frame would discard any row touched by any one of these NaNs, removing roughly 14 percent of all invoices and biasing revenue downward. A column by column policy is needed instead.


## 3. Cleaning policy

| Column | Action | Rationale |
|--------|--------|-----------|
| `Product line` | **Drop the row** | This column is the analysis pivot for product mix, segmentation, and basket rules. Imputing it would invent a product category that does not exist in the business. |
| `Customer type` | **Impute with mode** | Only two valid values (`Member`, `Normal`); using the mode preserves the existing class balance and keeps the row available for revenue and rating analysis. |
| `Unit price` | **Impute with median** | Median is robust to skew. Affected rows are very few (less than 1 percent). |
| `Quantity` | **Impute with median** | Same reasoning as `Unit price`. |
| `Date` | **Parse `%m/%d/%y`** | Two digit year format. Any unparseable row is dropped (none observed in this dataset). |

After imputation, the derived monetary columns (`Total`, `cogs`, `Tax 5%`, `gross income`) are recomputed from `Unit price` x `Quantity` so the row stays internally consistent. The whole policy is implemented in `src/data_loader.basic_clean`.


## 4. Apply and verify

In [5]:
df = basic_clean(raw)
print(f'rows kept: {len(df)} of {len(raw)} raw rows ({len(df) / len(raw):.1%})')
print(f'date range: {df["Date"].min().date()} to {df["Date"].max().date()}')

remaining = df.isna().sum()
assert remaining.sum() == 0, f'cleaning left NaNs: {remaining[remaining > 0].to_dict()}'
print('NaN per column after cleaning: all zero')
remaining.to_frame('remaining_nan')


rows kept: 960 of 1003 raw rows (95.7%)
date range: 2019-01-01 to 2019-03-30
NaN per column after cleaning: all zero


,remaining_nan
Invoice ID,0
Branch,0
City,0
Customer type,0
Gender,0
Product line,0
Unit price,0
Quantity,0
Tax 5%,0
Total,0


## 5. Does cleaning matter for downstream models?

A short experiment to confirm the policy is worth the trouble. The same rating prediction model (the one used in notebook 05) is trained on three differently cleaned versions of the data, and out of sample mean absolute error is reported. If cleaning did not matter, the three numbers would be similar.


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

def add_minimal_features(frame):
    f = frame.copy()
    f['Date'] = pd.to_datetime(f['Date'], errors='coerce')
    # Time may be string ('13:08') or datetime.time depending on source
    f['Hour'] = f['Time'].apply(lambda t: t.hour if hasattr(t, 'hour') else int(str(t).split(':')[0]) if pd.notna(t) else np.nan)
    f['IsWeekend'] = (f['Date'].dt.dayofweek >= 5).astype(int)
    f['IsMember'] = (f['Customer type'] == 'Member').astype(int)
    f['IsFemale'] = (f['Gender'] == 'Female').astype(int)
    return f

def score(frame, label):
    f = add_minimal_features(frame).dropna(subset=['Hour'])
    num = ['Unit price', 'Quantity', 'Total', 'Hour', 'IsWeekend', 'IsMember', 'IsFemale']
    cat = ['Branch', 'Product line', 'Payment']
    X, y = f[num + cat], f['Rating']
    pre = ColumnTransformer([('n', StandardScaler(), num),
                             ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=config.RANDOM_SEED)
    ridge = Pipeline([('p', pre), ('m', Ridge(alpha=1.0))]).fit(Xtr, ytr)
    gbr = Pipeline([('p', pre), ('m', GradientBoostingRegressor(random_state=config.RANDOM_SEED))]).fit(Xtr, ytr)
    return {
        'strategy': label,
        'rows': len(f),
        'ridge_mae': round(float(mean_absolute_error(yte, ridge.predict(Xte))), 4),
        'gbr_mae': round(float(mean_absolute_error(yte, gbr.predict(Xte))), 4),
    }

# Strategy A: drop every row that has any NaN
strat_a = raw.dropna().reset_index(drop=True)

# Strategy B: the policy adopted in this notebook
strat_b = df

# Strategy C: blind impute every NaN, including Product line
strat_c = raw.copy()
for c in strat_c.select_dtypes(include='number').columns:
    strat_c[c] = strat_c[c].fillna(strat_c[c].median())
for c in strat_c.select_dtypes(include='object').columns:
    if strat_c[c].isna().any():
        strat_c[c] = strat_c[c].fillna(strat_c[c].mode().iloc[0])

results = pd.DataFrame([
    score(strat_a, 'A. drop any NaN row'),
    score(strat_b, 'B. policy used here'),
    score(strat_c, 'C. blind impute everything'),
])
results


,strategy,rows,ridge_mae,gbr_mae
0,A. drop any NaN row,865,1.4657,1.5256
1,B. policy used here,960,1.4150,1.4377
2,C. blind impute everything,1003,1.5178,1.5684


Reading the table:

- **Strategy A (drop everything)**: simplest to explain but throws away the most data. Smaller training set, MAE worse.
- **Strategy B (the policy used here)**: keeps roughly 96 percent of the rows by being selective about what to drop and what to impute. Lowest MAE.
- **Strategy C (blind impute)**: keeps 100 percent of rows but invents Product line categories that did not exist in the business. The extra rows are noise, MAE is the worst of the three.

More rows is not automatically better. The policy that matches the meaning of each column wins.


## 6. Persist the cleaned table

In [7]:
out_path = config.DATA_PROCESSED / 'transactions_clean.parquet'
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)
print('written:', out_path)
print('shape:', df.shape)


written: D:\ZE5 PORTOFOLIO DS\Retail Analytics And Forecasting Platform\data\processed\transactions_clean.parquet
shape: (960, 17)


## Handover

The next notebook (`01_eda.ipynb`) loads `transactions_clean.parquet` directly. From this point onward, no notebook in the pipeline performs any cleaning. If a new data quality issue is discovered, the fix lives here, in `src/data_loader.basic_clean`, and propagates everywhere automatically by re-running this notebook.
